# Bank Marketing Prediction

Predicts whether a bank client will subscribe to a term deposit after a phone campaign (UCI Bank Marketing, 45,211 clients).


In [1]:

import io, zipfile, urllib.request
import pandas as pd
url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
zf = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(req).read()))
def find_csv(z, target):
    for n in z.namelist():
        if n.endswith(target): return z.read(n)
        if n.endswith(".zip"):
            try:
                r = find_csv(zipfile.ZipFile(io.BytesIO(z.read(n))), target)
                if r: return r
            except Exception: pass
d = pd.read_csv(io.BytesIO(find_csv(zf, "bank-full.csv")), sep=";")
print(d.shape); d.head()


(45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
y = (d["y"] == "yes").astype(int)
X = pd.get_dummies(d.drop(columns=["y"]), drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
models = {"Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          "Random Forest": RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)}
for name, m in models.items():
    m.fit(X_train, y_train); print(name, "accuracy:", round(m.score(X_test, y_test), 4))
best_name = max(models, key=lambda n: models[n].score(X_test, y_test))
best_model = models[best_name]
print("Best:", best_name, "| subscription rate:", round(float(y.mean()), 3))


Logistic Regression accuracy: 0.9015


Random Forest accuracy: 0.9077


Best: Random Forest | subscription rate: 0.117


In [3]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pred = best_model.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"{best_name} accuracy: {acc:.4f}")
print(classification_report(y_test, pred))
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i,j] > cm.max()/2 else "black")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=120); plt.show()


Random Forest accuracy: 0.9077
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      9981
           1       0.68      0.41      0.51      1322

    accuracy                           0.91     11303
   macro avg       0.80      0.69      0.73     11303
weighted avg       0.90      0.91      0.90     11303



## Result

See README.md for the verified accuracy. With an ~11.7% subscription rate, accuracy alone flatters the model — the notebook prints the full classification report so precision/recall for the 'yes' class are visible.
